# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshit5445/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)


**Lane:** Refresh / Content Opportunity Scoring

The feature vector uses February 2026 information available at the 2026-02-28 decision cutoff. March 2026 is used only for the outcome proxy `went_dark`.

Identifiers are used only for joining and grouping, then removed from the feature matrix.

## 0. Warehouse connection

In [8]:
import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = REL + "/fact_content_daily_performance"

FEB = "read_parquet('" + FACT + "/month=2026-02/*.parquet')"
MAR = "read_parquet('" + FACT + "/month=2026-03/*.parquet')"

print("Warehouse connection ready.")

Warehouse connection ready.


## 1. Build the feature vector

The feature vector contains February GSC and GA4 measures available at the decision cutoff. CTR and average position are engineered from February measurements. GSC and GA4 availability flags are retained as binary categorical indicators.

Unavailable GSC values contribute zero to the February aggregates while the GSC availability flag preserves whether GSC was measured. Unavailable GA4 sessions are filled with zero while the GA4 availability flag preserves availability. Missing February average position is filled with the observed February median.

Client and content identifiers are retained only while joining the feature window to the March outcome and are removed from `X`.

In [9]:
feb_features = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions ELSE 0 END) AS impressions_feb,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS clicks_feb,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_sum_position ELSE 0 END)
        / NULLIF(
            SUM(CASE WHEN gsc_data_available IS TRUE
                     THEN gsc_impressions ELSE 0 END), 0
        ) AS avg_position_feb,
        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_sessions ELSE 0 END) AS ga4_sessions_feb,
        MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
            AS gsc_available_feb,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            AS ga4_available_feb
    FROM {FEB}
    GROUP BY 1, 2
)
SELECT *,
       CASE
           WHEN impressions_feb > 0
           THEN clicks_feb / impressions_feb
           ELSE 0
       END AS ctr_feb
FROM feb
""").df()

march_labels = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS clicks_mar,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_gsc_days_mar
    FROM {MAR}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    clicks_mar,
    measured_gsc_days_mar,
    CASE WHEN clicks_mar = 0 THEN 1 ELSE 0 END AS went_dark
FROM march
WHERE measured_gsc_days_mar > 0
""").df()

frame = feb_features.merge(
    march_labels,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

position_median = frame["avg_position_feb"].median()
frame["avg_position_feb"] = frame["avg_position_feb"].fillna(position_median)
frame["ga4_sessions_feb"] = frame["ga4_sessions_feb"].fillna(0)

feature_columns = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "ga4_sessions_feb",
    "gsc_available_feb",
    "ga4_available_feb",
]

X = frame[feature_columns].copy()
y = frame["went_dark"].copy()

print("Feature vector shape:", X.shape)
print("Label rows:", len(y))
print("Positive labels:", int(y.sum()))
display(X.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (161539, 7)
Label rows: 161539
Positive labels: 98368


,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,ga4_sessions_feb,gsc_available_feb,ga4_available_feb
0,0.0,0.0,0.0,7.61194,0.0,0,0
1,3.0,0.0,0.0,9.00000,0.0,1,0
2,1.0,0.0,0.0,12.00000,0.0,1,0
3,2.0,0.0,0.0,35.50000,0.0,1,0
4,5.0,1.0,0.2,4.20000,0.0,1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Categorical | Available when? |
|---|---|---|---|---|
| `impressions_feb` | February measured GSC impressions | Unavailable GSC contributes 0 | No | By 2026-02-28 |
| `clicks_feb` | February measured GSC clicks | Unavailable GSC contributes 0 | No | By 2026-02-28 |
| `ctr_feb` | February clicks / impressions | 0 when impressions are 0 | No | By 2026-02-28 |
| `avg_position_feb` | February impression-weighted GSC position | Filled with February observed median | No | By 2026-02-28 |
| `ga4_sessions_feb` | February measured GA4 sessions | Unavailable GA4 filled with 0 | No | By 2026-02-28 |
| `gsc_available_feb` | Whether GSC was measured in February | Explicit binary indicator | Yes, binary | By 2026-02-28 |
| `ga4_available_feb` | Whether GA4 was measured in February | Explicit binary indicator | Yes, binary | By 2026-02-28 |

No client or content identifier is part of `X`.

In [10]:
feature_notes_check = pd.DataFrame({
    "feature": feature_columns,
    "dtype": [str(X[c].dtype) for c in feature_columns],
    "missing_after_fill": [int(X[c].isna().sum()) for c in feature_columns],
    "unique_values": [int(X[c].nunique(dropna=False)) for c in feature_columns],
})

display(feature_notes_check)

print("Rows:", len(X))
print("Columns:", X.shape[1])
print("Missing feature values:", int(X.isna().sum().sum()))

,feature,dtype,missing_after_fill,unique_values
0,impressions_feb,float64,0,10920
1,clicks_feb,float64,0,347
2,ctr_feb,float64,0,22013
3,avg_position_feb,float64,0,90521
4,ga4_sessions_feb,float64,0,287
5,gsc_available_feb,int32,0,2
6,ga4_available_feb,int32,0,2


Rows: 161539
Columns: 7
Missing feature values: 0


## 3. The leakage hunt

The deliberate leakage test uses `clicks_mar`, which is part of the March outcome window and directly defines `went_dark`.

The test checks whether the leaked field can reproduce the label. Because the proxy is defined as zero March clicks, `clicks_mar == 0` should reproduce the label exactly.

The honest feature matrix contains only February fields.

In [11]:
leaked_prediction = (frame["clicks_mar"] == 0).astype(int)
leak_accuracy = (leaked_prediction == frame["went_dark"]).mean()

print(f"Accuracy using leaked March clicks: {leak_accuracy:.3f}")

forbidden_in_X = [
    c for c in X.columns
    if c in {
        "clicks_mar",
        "went_dark",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_clicks_mar",
        "gsc_impressions_mar",
        "ga4_sessions_mar",
    }
]

print("Forbidden fields present in X:", forbidden_in_X)

assert forbidden_in_X == []
assert "clicks_mar" not in X.columns
assert "went_dark" not in X.columns
assert "client_hash_id" not in X.columns
assert "content_hash_id" not in X.columns

print("Leakage checks passed.")

Accuracy using leaked March clicks: 1.000
Forbidden fields present in X: []
Leakage checks passed.


## 4. What I excluded and why

- `client_hash_id` — identifier used only for joins and grouping.
- `content_hash_id` — identifier used only for joins.
- `clicks_mar` — future outcome field and direct source of the label.
- March GSC impressions and clicks — measured after the February decision cutoff.
- March GA4 sessions — measured after the February decision cutoff.
- `went_dark` — label, not a feature.
- `report_date` — source daily date, not a decision-time page feature.
- Client names, URLs, or other identifying metadata — excluded from the feature vector and outputs.

In [12]:
excluded_fields_check = pd.DataFrame({
    "field": [
        "client_hash_id",
        "content_hash_id",
        "clicks_mar",
        "gsc_impressions_mar",
        "gsc_clicks_mar",
        "ga4_sessions_mar",
        "went_dark",
        "report_date",
    ],
    "reason": [
        "Identifier; used only for joins/grouping.",
        "Identifier; used only for joins.",
        "Future label source.",
        "Future outcome-window field.",
        "Future outcome-window field.",
        "Future outcome-window field.",
        "Label, not a feature.",
        "Daily source date, not a decision-time page feature.",
    ],
})

display(excluded_fields_check)

print("Final feature columns:")
print(feature_columns)
print("Final feature matrix shape:", X.shape)

,field,reason
0,client_hash_id,Identifier; used only for joins/grouping.
1,content_hash_id,Identifier; used only for joins.
2,clicks_mar,Future label source.
3,gsc_impressions_mar,Future outcome-window field.
4,gsc_clicks_mar,Future outcome-window field.
5,ga4_sessions_mar,Future outcome-window field.
6,went_dark,"Label, not a feature."
7,report_date,"Daily source date, not a decision-time page fe..."


Final feature columns:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'ga4_sessions_feb', 'gsc_available_feb', 'ga4_available_feb']
Final feature matrix shape: (161539, 7)


## Self-check

- [x] Every section is filled with Markdown reasoning and supporting code.
- [x] Runtime → Run all completes without errors.
- [x] No client names, URLs, or private queries are included in the analysis output.
- [x] Claims use careful language: observed, measured, directional, and decision-support.
- [x] Future March outcome fields and identifiers are excluded from `X`.
- [x] The executed notebook is committed under `work/notebooks/w03_feature_leakage_check.ipynb`.